# Hypothesis Testing - Basketball Performance Analysis

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# load data
df = pd.read_csv("../data/processed/cleaned_basketball_data.csv")
print(f"Dataset: {df.shape}")
df.head()

Dataset: (10000, 37)


,id,year,rank,school,games,wins,losses,win_percentage,conference_wins,conference_losses,...,offensive_rebounds,total_rebounds,assists,steals,blocks,turnovers,personal_fouls,points,opponent_points,simple_rating
0,1,2021,1,Villanova,38,30,8,0.789,16,4,...,10.3,34.8,11.9,6.0,2.2,9.9,14.9,71.7,62.7,19.31
1,2,2021,2,Providence,33,27,6,0.818,14,3,...,10.5,37.6,13.2,5.0,3.7,11.4,16.0,71.5,66.2,13.08
2,3,2021,3,UConn,33,23,10,0.697,13,6,...,13.8,40.4,14.0,5.9,6.4,11.8,16.8,74.8,65.3,16.40
3,4,2021,4,Creighton,35,23,12,0.657,12,7,...,9.6,38.1,13.3,5.5,4.3,14.1,13.6,69.2,66.4,11.34
4,5,2021,5,Marquette,32,19,13,0.594,11,8,...,7.8,34.8,16.0,7.8,5.2,12.4,17.4,74.0,71.6,11.36


In [3]:
# Compute derived metrics inline for hypothesis testing
# These will be formally created in notebook 04 (Feature Engineering)

df["assist_turnover_ratio"] = np.where(df["turnovers"] == 0, 0, df["assists"] / df["turnovers"])
df["home_win_pct"] = np.where((df["home_wins"] + df["home_losses"]) == 0, 0.5, df["home_wins"] / (df["home_wins"] + df["home_losses"]))
df["away_win_pct"] = np.where((df["away_wins"] + df["away_losses"]) == 0, 0.5, df["away_wins"] / (df["away_wins"] + df["away_losses"]))
df["home_advantage"] = df["home_win_pct"] - df["away_win_pct"]

print(f"Computed 3 exploratory features inline for hypothesis testing")

Computed 3 exploratory features inline for hypothesis testing


## Hypothesis 1: The Ball Control Hypothesis

- **Null Hypothesis (H₀):** Teams with high ATO ratio have the same win % as teams with low ATO ratio
- **Alternative Hypothesis (H₁):** Teams with high ATO ratio have significantly higher win %
- **Test:** Two-Sample Independent T-Test

In [4]:
# split teams at median assist/turnover ratio
median_ratio = df['assist_turnover_ratio'].median()
high_ratio = df[df['assist_turnover_ratio'] >= median_ratio]
low_ratio = df[df['assist_turnover_ratio'] < median_ratio]

print(f"High A/TO ratio teams: {len(high_ratio)}")
print(f"Low A/TO ratio teams: {len(low_ratio)}")
print(f"\nHigh ratio avg win %: {high_ratio['win_percentage'].mean():.4f}")
print(f"Low ratio avg win %: {low_ratio['win_percentage'].mean():.4f}")

# independent t-test
t_stat, p_value = stats.ttest_ind(high_ratio['win_percentage'], low_ratio['win_percentage'])

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\nREJECT H0")
else:
    print("\nFAIL TO REJECT H0")

High A/TO ratio teams: 5000
Low A/TO ratio teams: 5000

High ratio avg win %: 0.6048
Low ratio avg win %: 0.5956

t-statistic: 2.4529
p-value: 0.0142

REJECT H0


## Hypothesis 2: Defense Wins Championships
- **Null Hypothesis (H₀):** Elite defensive teams (top 25%) have the same win % as the rest of the league
- **Alternative Hypothesis (H₁):** Elite defensive teams have significantly higher win %
- **Test:** Two-Sample Independent T-Test

In [5]:
# top 25% defensive teams (lower is better)
defense_threshold = df['defensive_rating'].quantile(0.25)
elite_defense = df[df['defensive_rating'] <= defense_threshold]
rest = df[df['defensive_rating'] > defense_threshold]

print(f"Elite defense (top 25%): {len(elite_defense)} teams")
print(f"Rest of league: {len(rest)} teams")
print(f"Threshold: {defense_threshold:.2f}")
print(f"\nElite defense avg win %: {elite_defense['win_percentage'].mean():.4f}")
print(f"Rest avg win %: {rest['win_percentage'].mean():.4f}")

# independent t-test
t_stat, p_value = stats.ttest_ind(elite_defense['win_percentage'], rest['win_percentage'])

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\nREJECT H0")
else:
    print("\nFAIL TO REJECT H0")

Elite defense (top 25%): 2565 teams
Rest of league: 7435 teams
Threshold: 95.20

Elite defense avg win %: 0.6053
Rest avg win %: 0.5985

t-statistic: 1.5767
p-value: 0.1149

FAIL TO REJECT H0


## Hypothesis 3: Home Court Advantage
- **Null Hypothesis (H₀):** Home win % equals away win %
- **Alternative Hypothesis (H₁):** Home win % is significantly higher than away win %
- **Test:** Paired T-Test (same teams in two conditions)

In [6]:
# compare home vs away for same teams
print(f"Average home win %: {df['home_win_pct'].mean():.4f}")
print(f"Average away win %: {df['away_win_pct'].mean():.4f}")
print(f"Average advantage: {df['home_advantage'].mean():.4f}")

# paired t-test
t_stat, p_value = stats.ttest_rel(df['home_win_pct'], df['away_win_pct'])

print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
    print("\nREJECT H0")
else:
    print("\nFAIL TO REJECT H0")

Average home win %: 0.6706
Average away win %: 0.4242
Average advantage: 0.2464

t-statistic: 80.1518
p-value: 0.0000

REJECT H0
